# 72. 예측 봉인 뒤 추가 CN7 정답 개봉
71번예측·프로토콜해시확인후에만추가CN7정답을개봉한다. 중복키에quality/Reason충돌이있으면사전규칙대로별도계수한다. 가스0개이면재현율/F1은평가불가. 모델선택/임계값변경을이정답으로하지않는다. RG3개선독립증명으로바꾸지않는다.


In [ ]:
from pathlib import Path
import hashlib,numpy as np,pandas as pd
from sklearn.metrics import confusion_matrix
root=Path.cwd()
if not (root/'data').exists():root=root.parent
tables=root/'reports/tables'
path=tables/'cn7_transfer1276_predictions_SEALED.csv'
seal=pd.read_csv(tables/'cn7_transfer1276_prediction_seal.csv').iloc[0]
assert hashlib.sha256(path.read_bytes()).hexdigest()==seal.prediction_sha256
assert hashlib.sha256((root/'reports/cn7_extra1276_locked_transfer_protocol.json').read_bytes()).hexdigest()==seal.protocol_sha256
p=pd.read_csv(path,parse_dates=['TimeStamp'])
raw=pd.read_csv(root/'data/original/labeled_data.csv',usecols=['EQUIP_CD','PART_NAME','TimeStamp','PassOrFail','Reason'])
raw=raw[raw.PART_NAME.str.contains('CN7',na=False)&raw.PART_NAME.str.contains('RH',na=False)].copy()
raw.TimeStamp=pd.to_datetime(raw.TimeStamp)
key=['EQUIP_CD','PART_NAME','TimeStamp']
raw['_quality']=raw.PassOrFail.astype(str).str.strip().str.upper()
raw['_reason']=raw.Reason.fillna('').astype(str).str.strip()
counts=raw.groupby(key)[['_quality','_reason']].nunique()
conflict=counts.gt(1).any(axis=1).rename('label_conflict')
labels=raw.drop_duplicates(key).merge(conflict.reset_index(),on=key,validate='one_to_one')
g=p.merge(labels,on=key,validate='one_to_one')
good=g._quality.isin(['Y','0','0.0']);gas=g._quality.isin(['N','1','1.0'])&g._reason.str.contains('가스')
g['eligible']=(good|gas)&~g.label_conflict
g['actual_gas']=np.where(gas,1,np.where(good,0,np.nan))
g['status']=np.select([g.label_conflict,good,gas],['duplicate_label_conflict','normal','gas'],default='other_or_unknown')
rows=[]
for name in ['reference','primary','secondary']:
 q=g[g.eligible]
 yy=q.actual_gas.to_numpy(int);pr=q[name+'_prediction'].to_numpy(int)
 tn,fp,fn,tp=confusion_matrix(yy,pr,labels=[0,1]).ravel()
 n=int(yy.sum())
 rows.append(dict(model=name,test_keys=len(g),eligible=len(q),gas=n,normal=int((yy==0).sum()),excluded=int((~g.eligible).sum()),TP=int(tp),FP=int(fp),FN=int(fn),TN=int(tn),recall=tp/(tp+fn) if n else np.nan,F1=2*tp/(2*tp+fp+fn) if n else np.nan,gas_performance_evaluable=bool(n>0)))
m=pd.DataFrame(rows)
g.to_csv(tables/'cn7_transfer1276_unblinded_results.csv',index=False,encoding='utf-8-sig')
m.to_csv(tables/'cn7_transfer1276_metrics.csv',index=False,encoding='utf-8-sig')
print('BLIND ADDITIONAL CN7 label states')
print(g.status.value_counts().to_string())
print(m.to_string(index=False))
print('This holdout is consumed. No relabeling/pseudo-ground-truth or retuning on it may be called independent.')
